In [56]:
"""
cv_lab_1.py
Каждый результат сохраняется в папку "results"
и получает читаемое имя: <input_basename>__<method>.png
Единственный вызов OpenCV: cv2.imread()
"""
import sys
import os
import numpy as np
import cv2            # только для imread
from PIL import Image, ImageDraw


# ---------- Утилиты ----------
def to_rgb(img_bgr):
    """Перевод BGR (cv2.imread) -> RGB, uint8"""
    return img_bgr[..., ::-1].copy()


def ensure_out_dir(out_dir):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)


def save_and_show(img_array, out_dir, base_name, method_name, show=True):
    """
    Сохранить изображение в out_dir с именем <base_name>__<method_name>.png
    и открыть его (PIL.Image.show).
    Поддерживает 2D (grayscale) и 3D (RGB).
    """
    ensure_out_dir(out_dir)

    if img_array.dtype != np.uint8:
        arr = np.clip(img_array, 0, 255).astype(np.uint8)
    else:
        arr = img_array

    safe_method = method_name.replace(" ", "_").replace("/", "_")
    filename = f"{base_name}__{safe_method}.png"
    path = os.path.join(out_dir, filename)

    pil_img = Image.fromarray(arr)
    pil_img.save(path)
    print(f"[saved] {path}")

    if show:
        try:
            pil_img.show()
        except Exception as e:
            print(f"[warning] Не удалось показать изображение: {e}")


def pad_reflect(img, pad_h, pad_w):
    """Reflect padding (mirror) для 2D или 3D массива."""
    if img.ndim == 2:
        return np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    elif img.ndim == 3:
        return np.pad(img, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='reflect')
    else:
        raise ValueError("Unsupported ndim for padding")


# ---------- Фильтры ----------
def median_filter_gray(img_gray, ksize=3):
    assert ksize % 2 == 1
    h, w = img_gray.shape
    ph = ksize // 2
    padded = pad_reflect(img_gray, ph, ph)
    out = np.empty_like(img_gray)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+ksize, j:j+ksize].ravel()
            out[i, j] = np.median(window)
    return out


def median_filter_rgb(img_rgb, ksize=3):
    channels = []
    for c in range(3):
        channels.append(median_filter_gray(img_rgb[..., c], ksize))
    return np.stack(channels, axis=2)


def gaussian_kernel(ksize=5, sigma=1.0):
    assert ksize % 2 == 1
    ax = np.arange(-ksize//2 + 0.0, ksize//2 + 1.0)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    kernel = kernel / np.sum(kernel)
    return kernel


def convolve_gray(img_gray, kernel):
    k = kernel.shape[0]
    ph = k // 2
    h, w = img_gray.shape
    padded = pad_reflect(img_gray, ph, ph).astype(np.float64)
    out = np.zeros((h, w), dtype=np.float64)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+k, j:j+k]
            out[i, j] = np.sum(window * kernel)
    return np.clip(out, 0, 255).astype(np.uint8)


def gaussian_filter_rgb(img_rgb, ksize=5, sigma=1.0):
    kern = gaussian_kernel(ksize, sigma)
    chans = []
    for c in range(3):
        chans.append(convolve_gray(img_rgb[..., c], kern))
    return np.stack(chans, axis=2)


# ---------- Морфологические операции ----------
def erosion_gray(img_gray, se=None):
    if se is None:
        se = np.ones((3,3), dtype=bool)
    k = se.shape[0]
    ph = k // 2
    h, w = img_gray.shape
    padded = pad_reflect(img_gray, ph, ph)
    out = np.empty_like(img_gray)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+k, j:j+k]
            out[i,j] = np.min(window[se])
    return out


def dilation_gray(img_gray, se=None):
    if se is None:
        se = np.ones((3,3), dtype=bool)
    k = se.shape[0]
    ph = k // 2
    h, w = img_gray.shape
    padded = pad_reflect(img_gray, ph, ph)
    out = np.empty_like(img_gray)
    for i in range(h):
        for j in range(w):
            window = padded[i:i+k, j:j+k]
            out[i,j] = np.max(window[se])
    return out


def erosion_rgb(img_rgb, se=None):
    chans = []
    for c in range(3):
        chans.append(erosion_gray(img_rgb[..., c], se))
    return np.stack(chans, axis=2)


def dilation_rgb(img_rgb, se=None):
    chans = []
    for c in range(3):
        chans.append(dilation_gray(img_rgb[..., c], se))
    return np.stack(chans, axis=2)


# ---------- Пороговая бинаризация ----------
def rgb_to_gray(img_rgb):
    r = img_rgb[...,0].astype(np.float32)
    g = img_rgb[...,1].astype(np.float32)
    b = img_rgb[...,2].astype(np.float32)
    y = 0.299*r + 0.587*g + 0.114*b
    return np.clip(y, 0, 255).astype(np.uint8)


def threshold_gray(img_gray, thresh=128, maxval=255, inv=False):
    if not inv:
        out = (img_gray >= thresh).astype(np.uint8) * maxval
    else:
        out = (img_gray < thresh).astype(np.uint8) * maxval
    return out


def threshold_rgb(img_rgb, thresh=128, mode='per_channel'):
    if mode == 'per_channel':
        out = np.zeros_like(img_rgb)
        for c in range(3):
            out[..., c] = threshold_gray(img_rgb[..., c], thresh)
        return out
    elif mode == 'luminance':
        gray = rgb_to_gray(img_rgb)
        th = threshold_gray(gray, thresh)
        return np.stack([th, th, th], axis=2)
    else:
        raise ValueError("Unknown mode")


# ---------- Выравнивание гистограммы ----------
def equalize_hist_gray(img_gray):
    hist, _ = np.histogram(img_gray.flatten(), bins=256, range=(0,256))
    cdf = np.cumsum(hist).astype(np.float64)
    cdf_min = cdf[cdf > 0].min() if np.any(cdf > 0) else 0.0
    n = img_gray.size
    if n == cdf_min:
        lut = np.arange(256).astype(np.uint8)
    else:
        lut = np.round((cdf - cdf_min) / (n - cdf_min) * 255.0).astype(np.uint8)
    out = lut[img_gray]
    return out


def equalize_hist_rgb(img_rgb):
    arr = img_rgb.astype(np.float32)
    R = arr[...,0]; G = arr[...,1]; B = arr[...,2]
    Y  =  0.299*R + 0.587*G + 0.114*B
    Cb = 128 + (-0.168736*R - 0.331264*G + 0.5*B)
    Cr = 128 + (0.5*R - 0.418688*G - 0.081312*B)
    Y_eq = equalize_hist_gray(np.clip(Y,0,255).astype(np.uint8)).astype(np.float32)
    R2 = Y_eq + 1.402 * (Cr - 128)
    G2 = Y_eq - 0.344136 * (Cb - 128) - 0.714136 * (Cr - 128)
    B2 = Y_eq + 1.772 * (Cb - 128)
    rgb2 = np.stack([R2, G2, B2], axis=2)
    return np.clip(rgb2, 0, 255).astype(np.uint8)


# ---------- Визуализация гистограммы ----------
def compute_hist_gray(img_gray):
    """
    Вычисляет гистограмму grayscale-изображения.
    Возвращает массив длины 256: hist[v] = количество пикселей со значением v.
    """
    hist, _ = np.histogram(img_gray.flatten(), bins=256, range=(0, 256))
    return hist


def render_histogram_image(hist, title="", width=512, height=300,
                            bg=(255,255,255), fg=(30,30,30), bar_color=(70,130,180)):
    """
    Превращает массив гистограммы длины 256 в RGB-изображение (NumPy uint8).

    Параметры
    ---------
    hist       : np.ndarray, shape (256,) — гистограмма яркостей
    title      : str — подпись, которая будет напечатана в верхнем левом углу
    width      : int — ширина итогового изображения в пикселях
    height     : int — высота итогового изображения в пикселях
    bg         : (R,G,B) — цвет фона
    fg         : (R,G,B) — цвет осей, делений и текста
    bar_color  : (R,G,B) — цвет столбцов гистограммы
    """
    canvas = Image.new('RGB', (width, height), bg)
    draw = ImageDraw.Draw(canvas)

    ml, mr, mt, mb = 40, 15, 30, 30      # поля: left/right/top/bottom
    plot_w = width - ml - mr
    plot_h = height - mt - mb

    max_h = int(hist.max()) if hist.max() > 0 else 1

    # --- Заголовок ---
    if title:
        draw.text((ml, 4), title, fill=fg)

    # --- Оси ---
    # вертикальная
    draw.line([(ml, mt), (ml, mt + plot_h)], fill=fg, width=1)
    # горизонтальная
    draw.line([(ml, mt + plot_h), (ml + plot_w, mt + plot_h)], fill=fg, width=1)

    # --- Деления по оси X: 0, 64, 128, 192, 255 ---
    for v in [0, 64, 128, 192, 255]:
        px = ml + int(v / 255 * (plot_w - 1))
        draw.line([(px, mt + plot_h), (px, mt + plot_h + 4)], fill=fg, width=1)
        draw.text((px - 8, mt + plot_h + 6), str(v), fill=fg)

    # --- Деления по оси Y: 0%, 50%, 100% ---
    for frac, label in [(0.0, "0"), (0.5, "50%"), (1.0, "max")]:
        py = mt + plot_h - int(frac * (plot_h - 1))
        draw.line([(ml - 4, py), (ml, py)], fill=fg, width=1)
        draw.text((1, py - 5), label, fill=fg)

    # --- Столбики / вертикальные линии ---
    for x in range(256):
        value = hist[x]
        if value == 0:
            continue
        bar_h = int((value / max_h) * (plot_h - 1))
        px = ml + int(x / 255 * (plot_w - 1))
        y0 = mt + plot_h
        y1 = y0 - bar_h
        draw.line([(px, y0), (px, y1)], fill=bar_color, width=1)

    return np.array(canvas, dtype=np.uint8)


def save_histogram_gray(img_gray, out_dir, base_name, method_name,
                         title="", show=True):
    """
    Вычислить гистограмму grayscale-изображения,
    сохранить её PNG-рисунок в out_dir с именованием по стандарту проекта.
    """
    hist = compute_hist_gray(img_gray)
    hist_img = render_histogram_image(hist, title=title)
    save_and_show(hist_img, out_dir, base_name, method_name, show=show)


# ---------- Повороты на кратные 90 градусов ----------
def rotate_90_multiple(img, k=1):
    k = k % 4
    if k == 0:
        return img.copy()
    if img.ndim == 2:
        if k == 1:
            return np.flipud(np.transpose(img))
        if k == 2:
            return np.fliplr(np.flipud(img))
        if k == 3:
            return np.transpose(np.flipud(img))
    else:
        if k == 1:
            return np.flipud(np.transpose(img, (1,0,2)))
        if k == 2:
            return np.fliplr(np.flipud(img))
        if k == 3:
            return np.transpose(np.flipud(img), (1,0,2))


In [57]:
def bgr_to_rgb(img):
    return img[..., ::-1].copy()


def bgr_to_gray(img):
    return (
        0.114 * img[:, :, 0].astype(np.float64)
      + 0.587 * img[:, :, 1].astype(np.float64)
      + 0.299 * img[:, :, 2].astype(np.float64)
    ).astype(np.uint8)


def gray_to_rgb(img):
    return np.stack([img, img, img], axis=2)

# ============================================================
# STEP 2a. CLAHE - manual implementation
# ============================================================

def clahe_manual(gray, clip_limit=2.0, tile_size=8):
    """
    Contrast Limited Adaptive Histogram Equalization (manual).
    Each tile_size x tile_size block is equalized independently.
    The histogram is clipped at clip_limit*(n_pixels/256);
    the excess is redistributed uniformly across all 256 bins.
    """
    h, w = gray.shape
    out = np.zeros_like(gray, dtype=np.uint8)
    th = (h + tile_size - 1) // tile_size
    tw = (w + tile_size - 1) // tile_size
    for tr in range(th):
        for tc in range(tw):
            y0 = tr * tile_size
            y1 = min((tr + 1) * tile_size, h)
            x0 = tc * tile_size
            x1 = min((tc + 1) * tile_size, w)
            tile = gray[y0:y1, x0:x1].astype(np.int32)
            n_pix = tile.size
            hist = np.bincount(tile.ravel(), minlength=256).astype(np.float64)
            clip = clip_limit * n_pix / 256.0
            excess = np.maximum(0.0, hist - clip).sum()
            hist = np.minimum(hist, clip)
            hist += excess / 256.0
            cdf = np.cumsum(hist)
            cdf_min = cdf[cdf > 0][0]
            denom = max(float(n_pix) - cdf_min, 1.0)
            lut = np.round((cdf - cdf_min) / denom * 255.0).clip(0, 255).astype(np.uint8)
            out[y0:y1, x0:x1] = lut[tile]
    return out

# ============================================================
# STEP 1+2b. DoG KEYPOINT DETECTOR + ADAPTIVE NMS
# ============================================================

def gaussian_kernel_1d(sigma):
    radius = max(1, int(3.0 * sigma + 0.5))
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def gaussian_blur_manual(img, sigma):
    """Separable 1-D Gaussian: horizontal pass then vertical pass."""
    k = gaussian_kernel_1d(sigma)
    r = len(k) // 2
    img_f = img.astype(np.float64)
    padded = np.pad(img_f, [(0, 0), (r, r)], mode='reflect')
    h_out = sum(k[i] * padded[:, i:i + img_f.shape[1]] for i in range(len(k)))
    padded = np.pad(h_out, [(r, r), (0, 0)], mode='reflect')
    return sum(k[i] * padded[i:i + h_out.shape[0], :] for i in range(len(k)))


def build_dog_pyramid(gray, sigmas=(1.0, 1.6, 2.5, 4.0, 6.3, 10.0)):
    """
    Difference-of-Gaussians pyramid.
    DoG approximates the scale-normalised Laplacian - an efficient blob detector.
    """
    blurs = [gaussian_blur_manual(gray, s) for s in sigmas]
    return [blurs[i + 1] - blurs[i] for i in range(len(blurs) - 1)]


def find_local_maxima(dog, threshold=4.0, min_dist=12):
    """
    Scan |DoG| with a sliding window of width 2*min_dist.
    Keep a pixel only if it is the unique maximum in its window
    AND exceeds the threshold.  Step = min_dist//2 overlaps windows.
    """
    h, w = dog.shape
    abs_dog = np.abs(dog)
    kps = []
    step = max(1, min_dist // 2)
    for y in range(min_dist, h - min_dist, step):
        for x in range(min_dist, w - min_dist, step):
            patch = abs_dog[y - min_dist:y + min_dist,
                            x - min_dist:x + min_dist]
            lmax = float(np.max(patch))
            center = float(abs_dog[y, x])
            if center == lmax and center > threshold:
                kps.append((x, y, center))
    return kps


def adaptive_nms(keypoints, target_n=400):
    """
    Adaptive NMS: binary search for the minimum suppression radius r*
    that keeps exactly target_n keypoints.
    Points sorted by descending response so stronger ones suppress weaker neighbours.
    """
    if len(keypoints) <= target_n:
        return keypoints
    kps = sorted(keypoints, key=lambda k: -k[2])
    lo = 1.0
    hi = 2000.0
    best = kps[:target_n]
    for _ in range(25):
        mid = (lo + hi) / 2.0
        selected = []
        suppressed = set()
        for i in range(len(kps)):
            if i in suppressed:
                continue
            selected.append(kps[i])
            for j in range(i + 1, len(kps)):
                if j not in suppressed:
                    dx = kps[j][0] - kps[i][0]
                    dy = kps[j][1] - kps[i][1]
                    if dx * dx + dy * dy < mid * mid:
                        suppressed.add(j)
        if len(selected) >= target_n:
            best = selected[:target_n]
            lo = mid
        else:
            hi = mid
    return best


def downsample_2x(gray):
    return gaussian_blur_manual(gray, 1.0)[::2, ::2]


def draw_keypoints_pil(rgb_img, keypoints):
    """Draw keypoints with PIL ellipses (no cv2.circle)."""
    pil = Image.fromarray(rgb_img)
    draw = ImageDraw.Draw(pil)
    for kp in keypoints:
        x = int(kp[0])
        y = int(kp[1])
        draw.ellipse([x - 6, y - 6, x + 6, y + 6], outline=(0, 255, 0), width=1)
        draw.ellipse([x - 2, y - 2, x + 2, y + 2], fill=(0, 200, 255))
    return np.array(pil)


# ============================================================
# STEP 3. SIFT DESCRIPTOR (fully manual)
# ============================================================

def compute_gradient(img):
    """Central-difference gradient (2nd-order accurate)."""
    f = img.astype(np.float64)
    gx = np.zeros_like(f)
    gy = np.zeros_like(f)
    gx[:, 1:-1] = (f[:, 2:] - f[:, :-2]) / 2.0
    gx[:, 0] = f[:, 1] - f[:, 0]
    gx[:, -1] = f[:, -1] - f[:, -2]
    gy[1:-1, :] = (f[2:, :] - f[:-2, :]) / 2.0
    gy[0, :] = f[1, :] - f[0, :]
    gy[-1, :] = f[-1, :] - f[-2, :]
    return np.sqrt(gx ** 2 + gy ** 2), np.arctan2(gy, gx)


def dominant_orientation(mag_patch, ori_patch, n_bins=36):
    """
    Weighted gradient-orientation histogram of a patch.
    Weight = gradient magnitude.  Returns angle of the tallest bin (radians).
    """
    hist = np.zeros(n_bins)
    for angle, w in zip(ori_patch.ravel(), mag_patch.ravel()):
        b = int((angle + np.pi) / (2.0 * np.pi) * n_bins) % n_bins
        hist[b] += w
    return (float(np.argmax(hist)) / float(n_bins)) * 2.0 * np.pi - np.pi


def sift_descriptor_manual(eq, x, y, patch_size=16, n_cells=4, n_ori_bins=8):
    """
    Manual 128-D SIFT descriptor.

    1. Extract patch_size x patch_size patch (reflect-padded).
    2. Compute gradient magnitude/orientation.
    3. Find dominant orientation; subtract it (rotation invariance).
    4. Divide into n_cells x n_cells sub-cells; build n_ori_bins histogram each.
    5. Concatenate -> 128-D vector.
    6. L2-normalise -> clip at 0.2 -> L2-normalise again.
    """
    half = patch_size // 2
    pad = half + 2
    padded = np.pad(eq.astype(np.float64), pad, mode='reflect')
    cx = x + pad
    cy = y + pad
    patch = padded[cy - half:cy + half, cx - half:cx + half]
    if patch.shape != (patch_size, patch_size):
        return None
    mag, ori = compute_gradient(patch)
    dom_ori = dominant_orientation(mag, ori)
    ori_rel = ori - dom_ori
    cell = patch_size // n_cells
    desc = []
    for r in range(n_cells):
        for c in range(n_cells):
            m_cell = mag[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]
            o_cell = ori_rel[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]
            hist = np.zeros(n_ori_bins)
            for angle, weight in zip(o_cell.ravel(), m_cell.ravel()):
                b = int((angle + np.pi) / (2.0 * np.pi) * n_ori_bins) % n_ori_bins
                hist[b] += weight
            desc.extend(hist.tolist())
    desc = np.array(desc, dtype=np.float32)
    n = float(np.linalg.norm(desc))
    if n > 1e-6:
        desc = desc / n
    desc = np.clip(desc, 0.0, 0.2)
    n2 = float(np.linalg.norm(desc))
    if n2 > 1e-6:
        desc = desc / n2
    return desc


# ============================================================
# STEP 4. DESCRIPTOR MATCHING (Lowe ratio test, manual BF)
# ============================================================

def match_descriptors_manual(desc1, desc2, ratio_thresh=0.80):
    """
    Brute-force nearest-neighbour matching + Lowe ratio test (2004).
    Match (i, j1) accepted iff dist(i,j1) < ratio_thresh * dist(i,j2).
    """
    if len(desc1) == 0 or len(desc2) == 0:
        return []
    diff = desc1[:, np.newaxis, :] - desc2[np.newaxis, :, :]
    dist_mat = np.sqrt((diff ** 2).sum(axis=2))
    matches = []
    for i in range(len(desc1)):
        row = dist_mat[i]
        idx = np.argsort(row)
        if len(idx) >= 2 and row[idx[0]] < ratio_thresh * row[idx[1]]:
            matches.append((i, int(idx[0]), float(row[idx[0]])))
    return matches


def draw_matches_manual(img1, img2, kps1, kps2, matches, max_draw=40):
    """Visualise correspondences with PIL (replaces cv2.drawMatches)."""
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    canvas = Image.new('RGB', (w1 + w2, max(h1, h2)))
    canvas.paste(Image.fromarray(img1), (0, 0))
    canvas.paste(Image.fromarray(img2), (w1, 0))
    draw = ImageDraw.Draw(canvas)
    palette = [
        (255, 80, 80), (80, 255, 80), (80, 130, 255), (255, 210, 0),
        (0, 210, 255), (210, 0, 255), (255, 140, 0), (0, 255, 160),
    ]
    for idx, match in enumerate(matches[:max_draw]):
        i1 = match[0]
        i2 = match[1]
        x1 = int(kps1[i1][0])
        y1 = int(kps1[i1][1])
        x2 = int(kps2[i2][0]) + w1
        y2 = int(kps2[i2][1])
        c = palette[idx % len(palette)]
        draw.ellipse([x1 - 5, y1 - 5, x1 + 5, y1 + 5], outline=c, width=2)
        draw.ellipse([x2 - 5, y2 - 5, x2 + 5, y2 + 5], outline=c, width=2)
        draw.line([(x1, y1), (x2, y2)], fill=c, width=1)
    return canvas




In [58]:
def bgr_to_rgb(img):
    return img[..., ::-1].copy()


def bgr_to_gray(img):
    return (
        0.114 * img[:, :, 0].astype(np.float64)
      + 0.587 * img[:, :, 1].astype(np.float64)
      + 0.299 * img[:, :, 2].astype(np.float64)
    ).astype(np.uint8)


def gray_to_rgb(img):
    return np.stack([img, img, img], axis=2)


def save_grid(imgs, title, fname, cmap=None, labels=None):
    n = len(imgs)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.2))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    axes = axes.ravel()
    for i, img in enumerate(imgs):
        if cmap:
            axes[i].imshow(img, cmap=cmap)
        else:
            disp = img if img.ndim == 3 else gray_to_rgb(img)
            axes[i].imshow(disp)
        lbl = labels[i] if labels else ('Frame ' + str(i + 1))
        axes[i].set_title(lbl, fontsize=9)
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=110, bbox_inches='tight')
    plt.close()
    print('  Saved: output/' + fname)



def clahe_manual(gray, clip_limit=2.0, tile_size=8):
    """
    Contrast Limited Adaptive Histogram Equalization (manual).
    Each tile_size x tile_size block is equalized independently.
    The histogram is clipped at clip_limit*(n_pixels/256);
    the excess is redistributed uniformly across all 256 bins.
    """
    h, w = gray.shape
    out = np.zeros_like(gray, dtype=np.uint8)
    th = (h + tile_size - 1) // tile_size
    tw = (w + tile_size - 1) // tile_size
    for tr in range(th):
        for tc in range(tw):
            y0 = tr * tile_size
            y1 = min((tr + 1) * tile_size, h)
            x0 = tc * tile_size
            x1 = min((tc + 1) * tile_size, w)
            tile = gray[y0:y1, x0:x1].astype(np.int32)
            n_pix = tile.size
            hist = np.bincount(tile.ravel(), minlength=256).astype(np.float64)
            clip = clip_limit * n_pix / 256.0
            excess = np.maximum(0.0, hist - clip).sum()
            hist = np.minimum(hist, clip)
            hist += excess / 256.0
            cdf = np.cumsum(hist)
            cdf_min = cdf[cdf > 0][0]
            denom = max(float(n_pix) - cdf_min, 1.0)
            lut = np.round((cdf - cdf_min) / denom * 255.0).clip(0, 255).astype(np.uint8)
            out[y0:y1, x0:x1] = lut[tile]
    return out


# ============================================================
# STEP 1+2b. DoG KEYPOINT DETECTOR + ADAPTIVE NMS
# ============================================================

def gaussian_kernel_1d(sigma):
    radius = max(1, int(3.0 * sigma + 0.5))
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def gaussian_blur_manual(img, sigma):
    """Separable 1-D Gaussian: horizontal pass then vertical pass."""
    k = gaussian_kernel_1d(sigma)
    r = len(k) // 2
    img_f = img.astype(np.float64)
    padded = np.pad(img_f, [(0, 0), (r, r)], mode='reflect')
    h_out = sum(k[i] * padded[:, i:i + img_f.shape[1]] for i in range(len(k)))
    padded = np.pad(h_out, [(r, r), (0, 0)], mode='reflect')
    return sum(k[i] * padded[i:i + h_out.shape[0], :] for i in range(len(k)))


def build_dog_pyramid(gray, sigmas=(1.0, 1.6, 2.5, 4.0, 6.3, 10.0)):
    """
    Difference-of-Gaussians pyramid.
    DoG approximates the scale-normalised Laplacian - an efficient blob detector.
    """
    blurs = [gaussian_blur_manual(gray, s) for s in sigmas]
    return [blurs[i + 1] - blurs[i] for i in range(len(blurs) - 1)]


def find_local_maxima(dog, threshold=4.0, min_dist=12):
    """
    Scan |DoG| with a sliding window of width 2*min_dist.
    Keep a pixel only if it is the unique maximum in its window
    AND exceeds the threshold.  Step = min_dist//2 overlaps windows.
    """
    h, w = dog.shape
    abs_dog = np.abs(dog)
    kps = []
    step = max(1, min_dist // 2)
    for y in range(min_dist, h - min_dist, step):
        for x in range(min_dist, w - min_dist, step):
            patch = abs_dog[y - min_dist:y + min_dist,
                            x - min_dist:x + min_dist]
            lmax = float(np.max(patch))
            center = float(abs_dog[y, x])
            if center == lmax and center > threshold:
                kps.append((x, y, center))
    return kps


def adaptive_nms(keypoints, target_n=400):
    """
    Adaptive NMS: binary search for the minimum suppression radius r*
    that keeps exactly target_n keypoints.
    Points sorted by descending response so stronger ones suppress weaker neighbours.
    """
    if len(keypoints) <= target_n:
        return keypoints
    kps = sorted(keypoints, key=lambda k: -k[2])
    lo = 1.0
    hi = 2000.0
    best = kps[:target_n]
    for _ in range(25):
        mid = (lo + hi) / 2.0
        selected = []
        suppressed = set()
        for i in range(len(kps)):
            if i in suppressed:
                continue
            selected.append(kps[i])
            for j in range(i + 1, len(kps)):
                if j not in suppressed:
                    dx = kps[j][0] - kps[i][0]
                    dy = kps[j][1] - kps[i][1]
                    if dx * dx + dy * dy < mid * mid:
                        suppressed.add(j)
        if len(selected) >= target_n:
            best = selected[:target_n]
            lo = mid
        else:
            hi = mid
    return best


def downsample_2x(gray):
    return gaussian_blur_manual(gray, 1.0)[::2, ::2]


def draw_keypoints_pil(rgb_img, keypoints):
    """Draw keypoints with PIL ellipses (no cv2.circle)."""
    pil = Image.fromarray(rgb_img)
    draw = ImageDraw.Draw(pil)
    for kp in keypoints:
        x = int(kp[0])
        y = int(kp[1])
        draw.ellipse([x - 6, y - 6, x + 6, y + 6], outline=(0, 255, 0), width=1)
        draw.ellipse([x - 2, y - 2, x + 2, y + 2], fill=(0, 200, 255))
    return np.array(pil)


# ============================================================
# STEP 3. SIFT DESCRIPTOR (fully manual)
# ============================================================

def compute_gradient(img):
    """Central-difference gradient (2nd-order accurate)."""
    f = img.astype(np.float64)
    gx = np.zeros_like(f)
    gy = np.zeros_like(f)
    gx[:, 1:-1] = (f[:, 2:] - f[:, :-2]) / 2.0
    gx[:, 0] = f[:, 1] - f[:, 0]
    gx[:, -1] = f[:, -1] - f[:, -2]
    gy[1:-1, :] = (f[2:, :] - f[:-2, :]) / 2.0
    gy[0, :] = f[1, :] - f[0, :]
    gy[-1, :] = f[-1, :] - f[-2, :]
    return np.sqrt(gx ** 2 + gy ** 2), np.arctan2(gy, gx)


def dominant_orientation(mag_patch, ori_patch, n_bins=36):
    """
    Weighted gradient-orientation histogram of a patch.
    Weight = gradient magnitude.  Returns angle of the tallest bin (radians).
    """
    hist = np.zeros(n_bins)
    for angle, w in zip(ori_patch.ravel(), mag_patch.ravel()):
        b = int((angle + np.pi) / (2.0 * np.pi) * n_bins) % n_bins
        hist[b] += w
    return (float(np.argmax(hist)) / float(n_bins)) * 2.0 * np.pi - np.pi


def sift_descriptor_manual(eq, x, y, patch_size=16, n_cells=4, n_ori_bins=8):
    """
    Manual 128-D SIFT descriptor.

    1. Extract patch_size x patch_size patch (reflect-padded).
    2. Compute gradient magnitude/orientation.
    3. Find dominant orientation; subtract it (rotation invariance).
    4. Divide into n_cells x n_cells sub-cells; build n_ori_bins histogram each.
    5. Concatenate -> 128-D vector.
    6. L2-normalise -> clip at 0.2 -> L2-normalise again.
    """
    half = patch_size // 2
    pad = half + 2
    padded = np.pad(eq.astype(np.float64), pad, mode='reflect')
    cx = x + pad
    cy = y + pad
    patch = padded[cy - half:cy + half, cx - half:cx + half]
    if patch.shape != (patch_size, patch_size):
        return None
    mag, ori = compute_gradient(patch)
    dom_ori = dominant_orientation(mag, ori)
    ori_rel = ori - dom_ori
    cell = patch_size // n_cells
    desc = []
    for r in range(n_cells):
        for c in range(n_cells):
            m_cell = mag[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]
            o_cell = ori_rel[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]
            hist = np.zeros(n_ori_bins)
            for angle, weight in zip(o_cell.ravel(), m_cell.ravel()):
                b = int((angle + np.pi) / (2.0 * np.pi) * n_ori_bins) % n_ori_bins
                hist[b] += weight
            desc.extend(hist.tolist())
    desc = np.array(desc, dtype=np.float32)
    n = float(np.linalg.norm(desc))
    if n > 1e-6:
        desc = desc / n
    desc = np.clip(desc, 0.0, 0.2)
    n2 = float(np.linalg.norm(desc))
    if n2 > 1e-6:
        desc = desc / n2
    return desc

# ============================================================
# STEP 4. DESCRIPTOR MATCHING (Lowe ratio test, manual BF)
# ============================================================

def match_descriptors_manual(desc1, desc2, ratio_thresh=0.80):
    """
    Brute-force nearest-neighbour matching + Lowe ratio test (2004).
    Match (i, j1) accepted iff dist(i,j1) < ratio_thresh * dist(i,j2).
    """
    if len(desc1) == 0 or len(desc2) == 0:
        return []
    diff = desc1[:, np.newaxis, :] - desc2[np.newaxis, :, :]
    dist_mat = np.sqrt((diff ** 2).sum(axis=2))
    matches = []
    for i in range(len(desc1)):
        row = dist_mat[i]
        idx = np.argsort(row)
        if len(idx) >= 2 and row[idx[0]] < ratio_thresh * row[idx[1]]:
            matches.append((i, int(idx[0]), float(row[idx[0]])))
    return matches


def draw_matches_manual(img1, img2, kps1, kps2, matches, max_draw=40):
    """Visualise correspondences with PIL (replaces cv2.drawMatches)."""
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    canvas = Image.new('RGB', (w1 + w2, max(h1, h2)))
    canvas.paste(Image.fromarray(img1), (0, 0))
    canvas.paste(Image.fromarray(img2), (w1, 0))
    draw = ImageDraw.Draw(canvas)
    palette = [
        (255, 80, 80), (80, 255, 80), (80, 130, 255), (255, 210, 0),
        (0, 210, 255), (210, 0, 255), (255, 140, 0), (0, 255, 160),
    ]
    for idx, match in enumerate(matches[:max_draw]):
        i1 = match[0]
        i2 = match[1]
        x1 = int(kps1[i1][0])
        y1 = int(kps1[i1][1])
        x2 = int(kps2[i2][0]) + w1
        y2 = int(kps2[i2][1])
        c = palette[idx % len(palette)]
        draw.ellipse([x1 - 5, y1 - 5, x1 + 5, y1 + 5], outline=c, width=2)
        draw.ellipse([x2 - 5, y2 - 5, x2 + 5, y2 + 5], outline=c, width=2)
        draw.line([(x1, y1), (x2, y2)], fill=c, width=1)
    return canvas


In [59]:
# lab3\lab2_functions.py
# Экспорт функций из Лаб. 2 для использования в Лаб. 3

import numpy as np


def gaussian_kernel_1d(sigma):
    radius = max(1, int(3 * sigma + 0.5))
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()

def gaussian_blur_manual(img, sigma):
    """Разделимая 2D свёртка с гауссовым ядром."""
    k = gaussian_kernel_1d(sigma)
    r = len(k) // 2
    img_f = img.astype(np.float64)
    padded = np.pad(img_f, [(0, 0), (r, r)], mode='reflect')
    h_out  = sum(k[i] * padded[:, i:i + img_f.shape[1]] for i in range(len(k)))
    padded = np.pad(h_out, [(r, r), (0, 0)], mode='reflect')
    v_out  = sum(k[i] * padded[i:i + img_f.shape[0], :] for i in range(len(k)))
    return v_out

def clahe_manual(gray, clip_limit=2.0, tile_size=8):
    """CLAHE — адаптивная эквализация гистограммы."""
    h, w = gray.shape
    output = np.zeros_like(gray, dtype=np.uint8)
    th = (h + tile_size - 1) // tile_size
    tw = (w + tile_size - 1) // tile_size
    for tr in range(th):
        for tc in range(tw):
            y0, y1 = tr*tile_size, min((tr+1)*tile_size, h)
            x0, x1 = tc*tile_size, min((tc+1)*tile_size, w)
            tile = gray[y0:y1, x0:x1].astype(np.int32)
            n_pixels = tile.size
            hist  = np.bincount(tile.ravel(), minlength=256).astype(np.float64)
            clip  = clip_limit * n_pixels / 256.0
            excess = np.maximum(0, hist - clip).sum()
            hist  = np.minimum(hist, clip)
            hist += excess / 256.0
            cdf     = np.cumsum(hist)
            cdf_min = cdf[cdf > 0][0]
            lut = np.round(
                (cdf - cdf_min) / max(n_pixels - cdf_min, 1) * 255
            ).clip(0, 255).astype(np.uint8)
            output[y0:y1, x0:x1] = lut[tile]
    return output

def compute_gradient(img):
    """Градиент через центральные конечные разности."""
    f  = img.astype(np.float64)
    gx = np.zeros_like(f); gy = np.zeros_like(f)
    gx[:, 1:-1] = (f[:, 2:] - f[:, :-2]) / 2
    gx[:, 0]    =  f[:, 1]  - f[:, 0]
    gx[:, -1]   =  f[:, -1] - f[:, -2]
    gy[1:-1, :] = (f[2:, :] - f[:-2, :]) / 2
    gy[0, :]    =  f[1, :]  - f[0, :]
    gy[-1, :]   =  f[-1, :] - f[-2, :]
    return np.sqrt(gx**2 + gy**2), np.arctan2(gy, gx)

In [60]:
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import sys

print("Рабочая директория:", os.getcwd())
print("Файлы:", os.listdir())

VIDEO_FILE   = "mouse_1.avi"
OUTPUT_DIR   = "lab3_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BG_THRESH      = 20
MIN_AREA       = 150
MAX_AREA_RATIO = 0.5
MAX_FRAMES = 5000   # все кадры видео (~6076 в mouse_1.avi)
FRAME_STEP = 1      # каждый кадр без пропусков
SCALE      = 0.5    # уменьшение для скорости


Рабочая директория: d:\Libraries\Documents\VScode\CV_labs_Plotnikov_8E21\Lab_3
Файлы: ['.venv', 'CV_lab_3_Plotnikov_8E21.ipynb', 'lab3_output', 'mouse_1.avi']


In [61]:

# ══════════════════════════════════════════════════════════════
# 3.1  ПОЛУЧЕНИЕ ВИДЕОПОТОКА
# ══════════════════════════════════════════════════════════════

def resize_nn(img, scale=0.5):
    h, w = img.shape[:2]
    nh, nw = int(h * scale), int(w * scale)
    yi = np.clip((np.arange(nh) / scale).astype(int), 0, h - 1)
    xi = np.clip((np.arange(nw) / scale).astype(int), 0, w - 1)
    return img[yi[:, None], xi[None, :], :] if img.ndim == 3 \
           else img[yi[:, None], xi[None, :]]


def read_video_frames(path, max_frames=5000, step=1, scale=0.5):
    """
    3.1  Преобразуем видеофайл в последовательность изображений.
         Пробегаемся по всему файлу и записываем кадры в массив —
         точно как в методичке (цикл while success == True).
    """
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Не удалось открыть: {path}")

    fps   = cap.get(cv2.CAP_PROP_FPS) or 20.0
    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"[3.1] {path}  |  {W}x{H}  |  {fps:.1f} fps  |  ~{total} кадров")

    frames_bgr, frames_gray = [], []
    i       = 1
    success = True

    # ── точная копия цикла из методички ──────────────────────
    # while (success == True):
    #     success, img = cap.read()
    #     img_grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #     imgs.append(img_grey)
    #     i = i + 1
    # ─────────────────────────────────────────────────────────
    while success and len(frames_bgr) < max_frames:
        success, frame = cap.read()
        if not success or frame is None:
            break

        if i % step == 0:
            small = resize_nn(frame, scale)
            # конвертация в серый через lb1.rgb_to_gray (BGR→RGB→gray)
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            frames_bgr.append(small)
            frames_gray.append(gray)

        if i % 100 == 0:
            print(f"  [3.1] считано кадров: {i}/{total}")

        i += 1

    cap.release()

    h2, w2 = frames_bgr[0].shape[:2]
    print(f"[3.1] Итого: {len(frames_bgr)} кадров  |  размер: {w2}x{h2}")

    # ── начальный кадр и кадр с движением (пункт 3.1) ────────
    fig, axarr = plt.subplots(1, 2, figsize=(12, 5))
    axarr[0].imshow(frames_bgr[0][..., ::-1])
    axarr[0].set_title("Начальный кадр (Фон)\nОпорный кадр — мышки нет")
    axarr[0].axis("off")
    mid = len(frames_bgr) // 3
    axarr[1].imshow(frames_bgr[mid][..., ::-1])
    axarr[1].set_title(f"Кадр с движением (#{mid})\nМышка видна в кадре")
    axarr[1].axis("off")
    plt.suptitle("3.1  Получение видеопотока — последовательность изображений",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "3_1_initial_frames.png"),
                dpi=120, bbox_inches='tight')
    plt.close()
    print("[3.1] → 3_1_initial_frames.png")

    return frames_bgr, frames_gray, fps / step, (w2, h2)



In [62]:

# ══════════════════════════════════════════════════════════════
# 3.2  АЛГОРИТМ ВЫЧИТАНИЯ ФОНА — два метода из методички
# ══════════════════════════════════════════════════════════════

def sr_mod_fon(arr_gray):
    """
    Метод 1: Статическое усреднение (sr_mod_fon из методички).
    bg[i,j] = среднее значение пикселя по всем кадрам.
    Векторизовано через numpy — без двойных циклов.
    """
    acc = np.zeros(arr_gray[0].shape, dtype=np.float64)
    for img in arr_gray:
        acc += img.astype(np.float64)
    bg = np.uint8(acc / len(arr_gray))
    print(f"[3.2] sr_mod_fon: фон построен по {len(arr_gray)} кадрам")
    return bg


def mod_fon_moments(arr_gray, bg_thresh=20, jump_thresh=30):
    """
    Метод 2: Обновляемый фон (mod_fon из методички).
    bg = 0.5*frame + 0.5*bg  (скользящее среднее).
    Возвращает финальный фон и список центроидов mom[].
    """
    h, w   = arr_gray[0].shape
    bg     = np.zeros((h, w), dtype=np.uint8)
    mom    = [None] * len(arr_gray)

    for k, frame in enumerate(arr_gray):
        # обновление фона
        bg = np.uint8(0.5 * frame.astype(np.float32)
                    + 0.5 * bg.astype(np.float32))

        # вычитание + шумоподавление (lb2) + порог (lb1)
        diff  = np.abs(frame.astype(np.int16) - bg.astype(np.int16))
        diff_f = gaussian_blur_manual(diff.astype(np.float64), sigma=1.0)
        thresh = threshold_gray(
            np.clip(diff_f, 0, 255).astype(np.uint8), thresh=bg_thresh)

        # центроид через моменты (numpy-аналог cv2.moments)
        m00 = float(thresh.sum()) / 255.0
        if m00 > 1e-5:
            ys, xs = np.where(thresh > 0)
            m10 = float((xs * thresh[ys, xs]).sum()) / 255.0
            m01 = float((ys * thresh[ys, xs]).sum()) / 255.0
            cx, cy = m10 / (m00 + 1e-5), m01 / (m00 + 1e-5)
        else:
            cx, cy = 0.0, 0.0

        # фильтрация выбросов (как в методичке)
        if cx == 0 and cy == 0 and k > 0 and mom[k-1] is not None:
            mom[k] = mom[k-1]
        elif k > 0 and mom[k-1] is not None:
            if abs(cx-mom[k-1][0]) > jump_thresh or \
               abs(cy-mom[k-1][1]) > jump_thresh:
                mom[k] = mom[k-1]
            else:
                mom[k] = (cx, cy)
        else:
            mom[k] = (cx, cy)

        if k % 50 == 0:
            print(f"  [mod_fon] {k}/{len(arr_gray)}  "
                  f"→ центроид=({mom[k][0]:.1f}, {mom[k][1]:.1f})")

    print("[3.2] mod_fon: обновляемый фон готов")
    return bg, mom


def save_backgrounds(bg_avg, bg_upd, out_dir):
    """Сохраняет оба фона в grayscale (как на образце)."""

    # Усреднённый фон — точно как изображение из примера
    cv2.imwrite(os.path.join(out_dir, "3_2_background_avg.png"), bg_avg)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(bg_avg, cmap='gray', vmin=0, vmax=255)
    ax.set_title("3.2 Модель фона — усреднение (sr_mod_fon)\n"
                 "Пустая арена без мышки. Светлое пятно — засветка лампы.\n"
                 "Слабые контуры = места, где мышка долго сидела неподвижно",
                 fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "3_2_background_avg_vis.png"),
                dpi=150, bbox_inches='tight')
    plt.close()

    # Обновляемый фон
    cv2.imwrite(os.path.join(out_dir, "3_2_background_upd.png"), bg_upd)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(bg_upd, cmap='gray', vmin=0, vmax=255)
    ax.set_title("3.2 Модель фона — обновляемый (mod_fon)\n"
                 "Скользящее среднее: bg = 0.5·frame + 0.5·bg.\n"
                 "Адаптируется к освещению, но медленно 'забывает' объекты",
                 fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "3_2_background_upd_vis.png"),
                dpi=150, bbox_inches='tight')
    plt.close()
    print("[3.2] → 3_2_background_avg_vis.png  |  3_2_background_upd_vis.png")



In [63]:

# ══════════════════════════════════════════════════════════════
# 3.3–3.5  ВЫЧИТАНИЕ ФОНА + ОЧИСТКА МАСКИ
# ══════════════════════════════════════════════════════════════

def subtract_and_clean(frame_gray, bg_gray, thresh=20):
    """
    3.3  Применить вычитание фона: diff = |frame - bg|
    3.4  Получить маску переднего плана (foreground mask)
    3.5  Очистить маску:
         - gaussian_blur через lb2.gaussian_blur_manual (шумоподавление)
         - бинаризация  через lb1.threshold_gray
    """
    diff   = np.abs(frame_gray.astype(np.int16)
                  - bg_gray.astype(np.int16))
    diff_f = gaussian_blur_manual(diff.astype(np.float64), sigma=1.0)
    mask   = threshold_gray(
        np.clip(diff_f, 0, 255).astype(np.uint8), thresh=thresh)
    return mask


# ══════════════════════════════════════════════════════════════
# 3.6  СВЯЗНЫЕ КОМПОНЕНТЫ (контуры движущихся объектов)
# ══════════════════════════════════════════════════════════════

def connected_components(binary_mask):
    """3.6  Найти связные компоненты — аналог cv2.connectedComponents."""
    h, w   = binary_mask.shape
    labels = np.zeros((h, w), dtype=np.int32)
    parent = [0]
    nl     = 1

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    parent.append(1)
    for y in range(h):
        for x in range(w):
            if binary_mask[y, x] == 0:
                continue
            nb = []
            if y > 0 and labels[y-1, x] > 0: nb.append(labels[y-1, x])
            if x > 0 and labels[y, x-1] > 0: nb.append(labels[y, x-1])
            if not nb:
                nl += 1
                parent.append(nl)
                labels[y, x] = nl
            else:
                m = min(nb)
                labels[y, x] = m
                for n in nb:
                    if n != m: union(m, n)
    for y in range(h):
        for x in range(w):
            if labels[y, x] > 0:
                labels[y, x] = find(labels[y, x])
    uniq = np.unique(labels); uniq = uniq[uniq > 0]
    remap = {old: i+1 for i, old in enumerate(uniq)}
    for old, new in remap.items():
        labels[labels == old] = new
    return labels, len(uniq)


# ══════════════════════════════════════════════════════════════
# 3.7  ФИЛЬТРАЦИЯ ОБЪЕКТОВ ПО РАЗМЕРУ/ФОРМЕ
# ══════════════════════════════════════════════════════════════

def filter_objects(label_map, n_labels, min_area=150, max_ratio=0.5):
    """3.7  Отфильтровать объекты: убрать шум (мелкие) и фон (крупные)."""
    h, w     = label_map.shape
    max_area = int(h * w * max_ratio)
    areas    = np.bincount(label_map.ravel(), minlength=n_labels + 1)
    objects  = []
    for lbl in range(1, n_labels + 1):
        a = int(areas[lbl])
        if a < min_area or a > max_area:
            continue
        ys, xs = np.where(label_map == lbl)
        objects.append({"label": lbl, "area": a,
                         "x0": int(xs.min()), "y0": int(ys.min()),
                         "x1": int(xs.max()), "y1": int(ys.max())})
    return objects


# ══════════════════════════════════════════════════════════════
# 3.8  ЦЕНТРОИД ЧЕРЕЗ МОМЕНТЫ
# ══════════════════════════════════════════════════════════════

def compute_centroid_moments(label_map, lbl):
    """
    3.8  Вычислить центроид через моменты изображения.
         cx = m10/m00,  cy = m01/m00  (как cv2.moments в методичке).
    """
    ys, xs = np.where(label_map == lbl)
    if len(xs) == 0:
        return None
    m00 = float(len(xs))
    cx  = float(xs.sum()) / m00
    cy  = float(ys.sum()) / m00
    return (cx, cy)



In [64]:

# ══════════════════════════════════════════════════════════════
# РИСОВАНИЕ ВРУЧНУЮ (без cv2.line / cv2.rectangle / cv2.circle)
# ══════════════════════════════════════════════════════════════

def draw_rect(img, x0, y0, x1, y1, color=(0, 255, 0), t=2):
    h, w = img.shape[:2]
    x0, y0 = max(0,x0), max(0,y0)
    x1, y1 = min(w-1,x1), min(h-1,y1)
    for d in range(t):
        if y0-d >= 0: img[y0-d, x0:x1+1] = color
        if y1+d <  h: img[y1+d, x0:x1+1] = color
        if x0-d >= 0: img[y0:y1+1, x0-d] = color
        if x1+d <  w: img[y0:y1+1, x1+d] = color


def draw_circle(img, cx, cy, r=5, color=(0, 0, 255)):
    h, w   = img.shape[:2]
    ys, xs = np.ogrid[:h, :w]
    img[(xs-int(cx))**2 + (ys-int(cy))**2 <= r**2] = color


def draw_line(img, pt1, pt2, color=(255, 180, 0)):
    x0, y0 = int(pt1[0]), int(pt1[1])
    x1, y1 = int(pt2[0]), int(pt2[1])
    h, w   = img.shape[:2]
    dx, dy = abs(x1-x0), abs(y1-y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    err = dx - dy
    while True:
        if 0 <= y0 < h and 0 <= x0 < w:
            img[y0, x0] = color
        if x0 == x1 and y0 == y1:
            break
        e2 = 2 * err
        if e2 > -dy: err -= dy; x0 += sx
        if e2 <  dx: err += dx; y0 += sy


In [65]:


# ══════════════════════════════════════════════════════════════
# 3.9  ПОСТРОЕНИЕ ТРАЕКТОРИИ — три отдельных файла
# ══════════════════════════════════════════════════════════════

def plot_trajectory(traj, video_size, fps, out_dir, prefix="3_9"):
    """3.9  Накопить координаты центроида и построить траекторию."""
    if len(traj) < 2:
        print("[3.9] Недостаточно точек."); return

    arr = np.array(traj)
    xs, ys = arr[:, 0], arr[:, 1]
    t   = np.arange(len(xs)) / fps
    spd = np.sqrt(np.diff(xs)**2 + np.diff(ys)**2) * fps
    W, H = video_size

    # ── 1. 2D траектория ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    fig.patch.set_facecolor('#111111')
    ax.set_facecolor("#3f3fa3")
    sc = ax.scatter(xs, ys, c=np.arange(len(xs)),
                    cmap='plasma', s=14, zorder=3)
    ax.plot(xs, ys, color='white', alpha=0.25, linewidth=0.8)
    ax.set_xlim(0, W); ax.set_ylim(H, 0)
    ax.set_title(
        "3.9 Траектория объекта (2D)\n"
        "Каждая точка = позиция центроида мышки.\n"
        "Цвет: синий=начало → жёлтый=конец",
        color='white', fontsize=9)
    ax.tick_params(colors='white')
    plt.colorbar(sc, ax=ax, label='кадр')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_traj_2d.png"),
                dpi=130, bbox_inches='tight')
    plt.close()

    # ── 2. X(t) и Y(t) ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor('#111111')
    ax.set_facecolor('#1a1a2e')
    ax.plot(t, xs, label='X(t)', color='cyan',  linewidth=1.2)
    ax.plot(t, ys, label='Y(t)', color='tomato', linewidth=1.2)
    ax.set_xlabel("Время (с)", color='white')
    ax.set_ylabel("Координата (пикс)", color='white')
    ax.set_title(
        "3.9 Координаты центроида по времени\n"
        "X(t) — горизонталь (cyan), Y(t) — вертикаль (red).\n"
        "Плоские участки = остановки мышки",
        color='white', fontsize=9)
    ax.tick_params(colors='white')
    ax.legend(); ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_traj_coords.png"),
                dpi=130, bbox_inches='tight')
    plt.close()

    # ── 3. Скорость ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.patch.set_facecolor('#111111')
    ax.set_facecolor('#1a1a2e')
    ax.plot(t[1:], spd, color='gold', linewidth=1.2)
    ax.set_xlabel("Время (с)", color='white')
    ax.set_ylabel("Скорость (пикс/с)", color='white')
    ax.set_title(
        "3.9 Скорость движения мышки\n"
        "Пики = резкие броски, нули = остановки.\n"
        f"Средняя: {spd.mean():.1f}  |  Макс: {spd.max():.1f} пикс/с",
        color='white', fontsize=9)
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{prefix}_traj_speed.png"),
                dpi=130, bbox_inches='tight')
    plt.close()

    print(f"[3.9] → {prefix}_traj_2d.png  |  "
          f"{prefix}_traj_coords.png  |  {prefix}_traj_speed.png")



In [66]:

# ══════════════════════════════════════════════════════════════
# 3.10  РЕЗУЛЬТАТЫ — каждый кадр отдельно
# ══════════════════════════════════════════════════════════════

def save_results_separate(display_frames, out_dir):
    """
    3.10  Отобразить результаты:
          оригинал + маска + bounding box + траектория.
          Каждый кадр и каждый тип — отдельный PNG.
    """
    row_names  = ["original", "mask", "result"]
    row_titles = [
        "Оригинал",
        "Маска переднего плана",
        "Объект + bounding box + траектория"
    ]
    row_desc = [
        "Исходный кадр видео (серый).\nМышка видна, но фон мешает.",
        "Бинарная маска: белое = движение.\n"
        "После gaussian blur (lb2) и порога (lb1) шум удалён.",
        "Результат: зелёный bbox + красный центроид\n"
        "+ цветная траектория (синий→красный)."
    ]
    cmaps = [None, 'hot', None]

    # каждый кадр × каждый тип
    for col, (orig, mask, res) in enumerate(display_frames):
        imgs = [orig, mask, res]
        fn   = col + 1
        for row in range(3):
            fig, ax = plt.subplots(figsize=(7, 5))
            img = imgs[row]
            ax.imshow(img[..., ::-1] if img.ndim == 3 else img,
                      cmap=cmaps[row])
            ax.set_title(f"{row_titles[row]}  —  кадр {fn}\n{row_desc[row]}",
                         fontsize=8)
            ax.axis("off")
            plt.tight_layout()
            fname = f"3_10_frame{fn:02d}_{row_names[row]}.png"
            plt.savefig(os.path.join(out_dir, fname),
                        dpi=120, bbox_inches='tight')
            plt.close()

    # сводные строки
    for row, (rname, rtitle, rdesc, cmap) in enumerate(
            zip(row_names, row_titles, row_desc, cmaps)):
        n    = len(display_frames)
        fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
        if n == 1: axes = [axes]
        for col, (orig, mask, res) in enumerate(display_frames):
            img = [orig, mask, res][row]
            axes[col].imshow(
                img[..., ::-1] if img.ndim == 3 else img, cmap=cmap)
            axes[col].set_title(f"кадр {col+1}")
            axes[col].axis("off")
        fig.suptitle(f"3.10  {rtitle}\n{rdesc}", fontsize=10)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"3_10_row_{rname}.png"),
                    dpi=120, bbox_inches='tight')
        plt.close()

    print(f"[3.10] → {len(display_frames)*3} отдельных PNG  "
          f"+ 3 сводных строки")



In [67]:

# ══════════════════════════════════════════════════════════════
# ГЛАВНАЯ ФУНКЦИЯ
# ══════════════════════════════════════════════════════════════

def process_video(video_path):

    # 3.1 ─ получение кадров
    frames_bgr, frames_gray, fps, size = read_video_frames(
        video_path, MAX_FRAMES, FRAME_STEP, SCALE)

    # 3.2a ─ статический фон (усреднение)
    bg_avg = sr_mod_fon(frames_gray)

    # 3.2b ─ обновляемый фон + центроиды
    print("[3.2] Запуск mod_fon (обновляемый фон)...")
    bg_upd, mom = mod_fon_moments(
        frames_gray, bg_thresh=BG_THRESH, jump_thresh=30)

    # Сохраняем оба фона в grayscale
    save_backgrounds(bg_avg, bg_upd, OUTPUT_DIR)

    # ── Основной цикл (статический фон) ──────────────────────
    traj           = []
    display_frames = []
    save_every     = max(1, len(frames_bgr) // 5)

    for idx, (frame_bgr, frame_gray) in enumerate(
            zip(frames_bgr, frames_gray)):
        if idx % 50 == 0:
            print(f"[3.3-3.8] кадр {idx}/{len(frames_bgr)}")

        # 3.3–3.5 вычитание фона + очистка маски
        mask = subtract_and_clean(frame_gray, bg_avg, thresh=BG_THRESH)

        # 3.6 связные компоненты
        labels, n_labels = connected_components(mask)

        # 3.7 фильтрация
        objs = filter_objects(labels, n_labels, MIN_AREA, MAX_AREA_RATIO)

        vis      = frame_bgr.copy()
        centroid = None

        if objs:
            main     = max(objs, key=lambda o: o["area"])
            # 3.8 центроид через моменты
            centroid = compute_centroid_moments(labels, main["label"])
            draw_rect(vis,
                      main["x0"], main["y0"],
                      main["x1"], main["y1"])

        if centroid:
            traj.append(centroid)
            draw_circle(vis, centroid[0], centroid[1], r=5)

        # 3.9 рисуем накопленную траекторию
        for i in range(1, len(traj)):
            a     = i / max(len(traj), 1)
            color = (int(255*(1-a)), 50, int(255*a))
            draw_line(vis, traj[i-1], traj[i], color=color)

        if idx % save_every == 0:
            display_frames.append(
                (frame_bgr.copy(), mask.copy(), vis.copy()))

    # 3.9 ─ графики траектории (статический фон)
    plot_trajectory(traj, size, fps, OUTPUT_DIR, prefix="3_9_avg")

    # 3.9 ─ траектория по обновляемому фону
    mom_ok = [(m[0], m[1]) for m in mom if m is not None]
    if len(mom_ok) > 2:
        plot_trajectory(mom_ok, size, fps, OUTPUT_DIR, prefix="3_9_upd")

    # 3.10 ─ отдельные изображения
    save_results_separate(display_frames, OUTPUT_DIR)

    print(f"\n[ИТОГ] Точек траектории: {len(traj)}")
    return traj, fps


def print_summary(traj, fps):
    print("\n" + "=" * 54)
    print("ВЫВОД")
    print("=" * 54)
    print(f"Всего точек траектории : {len(traj)}")
    if len(traj) < 2:
        print("Объект не обнаружен."); return
    xs  = np.array([p[0] for p in traj], dtype=float)
    ys  = np.array([p[1] for p in traj], dtype=float)
    spd = np.sqrt(np.diff(xs)**2 + np.diff(ys)**2) * fps
    print(f"Диапазон X             : {int(xs.min())} — {int(xs.max())} пикс.")
    print(f"Диапазон Y             : {int(ys.min())} — {int(ys.max())} пикс.")
    print(f"Средняя скорость       : {spd.mean():.1f} пикс/с")
    print(f"Максимальная скорость  : {spd.max():.1f} пикс/с")
    print(f"Результаты сохранены   : {os.path.abspath(OUTPUT_DIR)}")
    print("=" * 54)


In [68]:
# if __name__ == "__main__":
#     video = sys.argv[1] if len(sys.argv) > 1 else VIDEO_FILE
#     if not os.path.exists(video):
#         raise FileNotFoundError(f"Файл не найден: {video}")
#     traj, fps = process_video(video)
#     print_summary(traj, fps)

video_path = "D:\Libraries\Documents\VScode\CV_labs_Plotnikov_8E21\Lab_3\mouse_1.avi"  # Укажите здесь ваш файл
traj, fps = process_video(video_path)
print_summary(traj, fps)

[3.1] D:\Libraries\Documents\VScode\CV_labs_Plotnikov_8E21\Lab_3\mouse_1.avi  |  640x480  |  20.0 fps  |  ~6076 кадров
  [3.1] считано кадров: 100/6076
  [3.1] считано кадров: 200/6076
  [3.1] считано кадров: 300/6076
  [3.1] считано кадров: 400/6076
  [3.1] считано кадров: 500/6076
  [3.1] считано кадров: 600/6076
  [3.1] считано кадров: 700/6076
  [3.1] считано кадров: 800/6076
  [3.1] считано кадров: 900/6076
  [3.1] считано кадров: 1000/6076
  [3.1] считано кадров: 1100/6076
  [3.1] считано кадров: 1200/6076
  [3.1] считано кадров: 1300/6076
  [3.1] считано кадров: 1400/6076
  [3.1] считано кадров: 1500/6076
  [3.1] считано кадров: 1600/6076
  [3.1] считано кадров: 1700/6076
  [3.1] считано кадров: 1800/6076
  [3.1] считано кадров: 1900/6076
  [3.1] считано кадров: 2000/6076
  [3.1] считано кадров: 2100/6076
  [3.1] считано кадров: 2200/6076
  [3.1] считано кадров: 2300/6076
  [3.1] считано кадров: 2400/6076
  [3.1] считано кадров: 2500/6076
  [3.1] считано кадров: 2600/6076
  [3.1